<h1>Environment</h1>

In [ ]:
# Python
# python==3.10.19

# Deep learning framework (GPU, CUDA 12.8)
# torch==2.9.1+cu128

# Scientific computing and data processing
# numpy==2.2.5
# pandas==2.3.3

# Machine learning
# scikit-learn==1.7.2
# joblib==1.5.3

# Hyperparameter optimization
# optuna==4.6.0

# Visualization
# matplotlib==3.10.8

<h1>Import library</h1>

In [ ]:
import os
import warnings
import joblib
import optuna
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datetime import datetime

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    matthews_corrcoef,
    auc
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

try:
    import optuna.visualization as vis
    VISUALIZATION_AVAILABLE = True
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("plotly is not installed, Optuna visualization will be skipped")

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

print("All required libraries imported")

<h1>Load global configuration</h1>

In [ ]:
# ======================
# Paths to ESM embeddings
# ======================
EMB_PATH_POS = "/path/to/positive_esm_embedding"
EMB_PATH_NEG = "/path/to/negative_esm_embedding"

# ======================
# Paths to external ESM embeddings
# =====================
DEFAULT_EXT_POS_EMB_DIR = "/path/to/external_positive_esm_embedding"
DEFAULT_EXT_NEG_EMB_DIR = "/path/to/external_negative_esm_embedding"

# ======================
# Data partitioning
# ======================
N_BASE_MODELS = 5
N_FOLDS = 5
POS_TOTAL = 153
POS_POOL = 125          
POS_TEST = 28           
NEG_TOTAL = 653
NEG_POOL = 625          
NEG_TEST = 28           
NEG_GROUP_SIZE = 125    
FOLD_SIZE_POS = POS_POOL // N_FOLDS  
FOLD_SIZE_NEG = NEG_GROUP_SIZE // N_FOLDS 

# ======================
# Training configuration
# ======================
OPTUNA_TRIALS = 100
RANDOM_SEED = 42 # Can be set by oneself
MODEL_SAVE_DIR = "/path/to/model_save"
ESM_LAYER = 33

print("Configuration completed")

<h1>Utility functions</h1>

In [ ]:
def set_all_seeds(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def find_all_pt_files(root_dir):
    pt_files = []
    for current_root, _, files in os.walk(root_dir):
        for f in files:
            if f.endswith(".pt"):
                pt_files.append(os.path.join(current_root, f))
    return sorted(pt_files)

def check_pt_file_count(root_dir, expected_count, dataset_name="dataset"):
    pt_files = find_all_pt_files(root_dir)
    actual_count = len(pt_files)

    print(f"\nChecking {dataset_name}")
    print(f"   Path: {root_dir}")
    print(f"   Found .pt files: {actual_count}")
    print(f"   Expected files: {expected_count}")

    if expected_count is not None and actual_count != expected_count:
        raise ValueError(
            f"{dataset_name} .pt file count mismatch! actual={actual_count}, expected={expected_count}"
        )

    print(f"{dataset_name} file count is correct")
    return pt_files

def load_embeddings_from_dir(emb_dir, layer=33, expected_count=None, dataset_name=None):
    if dataset_name is None:
        dataset_name = os.path.basename(emb_dir)

    pt_paths = find_all_pt_files(emb_dir)

    if not pt_paths:
        raise FileNotFoundError(f"No .pt files found in directory {emb_dir} and its subdirectories!")

    print(f"\nLoading {dataset_name}")
    print(f"   Found .pt files: {len(pt_paths)}")

    if expected_count is not None and len(pt_paths) != expected_count:
        raise ValueError(
            f"{dataset_name} .pt file count mismatch! actual={len(pt_paths)}, expected={expected_count}"
        )

    embeddings = []
    valid_files = []

    for filepath in pt_paths:
        rel_name = os.path.relpath(filepath, emb_dir)
        try:
            data = torch.load(filepath, map_location="cpu", weights_only=False)
            emb = data["mean_representations"][layer]
            if isinstance(emb, torch.Tensor):
                emb = emb.cpu().numpy()
            embeddings.append(emb)
            valid_files.append(rel_name)
        except Exception as e:
            print(f"Skipping invalid file {rel_name}: {e}")

    if not embeddings:
        raise ValueError(f"No valid embedding data found in directory {emb_dir}!")

    arr = np.stack(embeddings)
    print(f"From {emb_dir}, loaded {len(arr)} embeddings, dim={arr.shape[1]}")
    return arr, valid_files

def load_all_esm_features(ext_pos_dir=None, ext_neg_dir=None,
                          ext_pos_expected=164, ext_neg_expected=164):

    set_all_seeds(RANDOM_SEED)

    if ext_pos_dir is None:
        ext_pos_dir = DEFAULT_EXT_POS_EMB_DIR
    if ext_neg_dir is None:
        ext_neg_dir = DEFAULT_EXT_NEG_EMB_DIR

    # Check file counts
    check_pt_file_count(EMB_PATH_POS, POS_TOTAL, "AFP Positive")
    check_pt_file_count(EMB_PATH_NEG, NEG_TOTAL, "Hard Negative")
    check_pt_file_count(ext_pos_dir, ext_pos_expected, "External Positive")
    if ext_neg_dir is not None:
        check_pt_file_count(ext_neg_dir, ext_neg_expected, "External Negative")

    # Load all embeddings
    X_all_pos, pos_files = load_embeddings_from_dir(
        EMB_PATH_POS, layer=ESM_LAYER, expected_count=POS_TOTAL, dataset_name="AFP Positive"
    )
    X_all_neg, neg_files = load_embeddings_from_dir(
        EMB_PATH_NEG, layer=ESM_LAYER, expected_count=NEG_TOTAL, dataset_name="Hard Negative"
    )

    X_ext_pos, ext_pos_files = load_embeddings_from_dir(
        ext_pos_dir, layer=ESM_LAYER, expected_count=ext_pos_expected, dataset_name="External Positive"
    )
    X_ext_neg, ext_neg_files = None, None
    if ext_neg_dir is not None:
        X_ext_neg, ext_neg_files = load_embeddings_from_dir(
            ext_neg_dir, layer=ESM_LAYER, expected_count=ext_neg_expected, dataset_name="External Negative"
        )

    # Shuffle (for reproducibility)
    np.random.seed(RANDOM_SEED)
    pos_idx = np.random.permutation(len(X_all_pos))
    neg_idx = np.random.permutation(len(X_all_neg))
    X_all_pos = X_all_pos[pos_idx]
    X_all_neg = X_all_neg[neg_idx]

    # 1) Fixed internal test set (28+28)
    X_pos_pool = X_all_pos[:POS_POOL]
    X_pos_test = X_all_pos[POS_POOL:POS_POOL + POS_TEST]

    X_neg_pool = X_all_neg[:NEG_POOL]
    X_neg_test = X_all_neg[NEG_POOL:NEG_POOL + NEG_TEST]

    X_test = np.concatenate([X_pos_test, X_neg_test], axis=0)
    y_test = np.concatenate([np.ones(len(X_pos_test)), np.zeros(len(X_neg_test))])

    # 2) Positive pool 125 -> 5 folds (25 each)
    pos_folds = [X_pos_pool[i*FOLD_SIZE_POS:(i+1)*FOLD_SIZE_POS] for i in range(N_FOLDS)]

    # 3) Negative pool 625 -> 5 groups * 125 (one group per base model)
    neg_groups = [X_neg_pool[i*NEG_GROUP_SIZE:(i+1)*NEG_GROUP_SIZE] for i in range(N_BASE_MODELS)]

    # Shuffle each group again, then split into 5 folds (25 each)
    neg_group_folds = []
    for i, g in enumerate(neg_groups):
        rng = np.random.RandomState(RANDOM_SEED + 100 + i)
        perm = rng.permutation(len(g))
        g = g[perm]
        folds = [g[j*FOLD_SIZE_NEG:(j+1)*FOLD_SIZE_NEG] for j in range(N_FOLDS)]
        neg_group_folds.append(folds)

    # 4) Build Train/Val for each base model (will train 5 models)
    base_splits = []
    for k in range(N_BASE_MODELS):
        # Pos: fold k as val, others as train
        X_pos_val = pos_folds[k]
        X_pos_train = np.concatenate([pos_folds[j] for j in range(N_FOLDS) if j != k], axis=0)

        # Neg: fold k of group k as val, others as train
        X_neg_val = neg_group_folds[k][k]
        X_neg_train = np.concatenate([neg_group_folds[k][j] for j in range(N_FOLDS) if j != k], axis=0)

        X_train_k = np.concatenate([X_pos_train, X_neg_train], axis=0)
        y_train_k = np.concatenate([np.ones(len(X_pos_train)), np.zeros(len(X_neg_train))])

        X_val_k = np.concatenate([X_pos_val, X_neg_val], axis=0)
        y_val_k = np.concatenate([np.ones(len(X_pos_val)), np.zeros(len(X_neg_val))])

        base_splits.append({
            "X_train": X_train_k,
            "y_train": y_train_k,
            "X_val": X_val_k,
            "y_val": y_val_k
        })

    print("\nData split result:")
    print(f"   Pos pool={len(X_pos_pool)} -> 5 folds * {FOLD_SIZE_POS}")
    print(f"   Neg pool={len(X_neg_pool)} -> 5 groups * {NEG_GROUP_SIZE} -> each 5 folds * {FOLD_SIZE_NEG}")
    print(f"   Internal Test: Pos={len(X_pos_test)}, Neg={len(X_neg_test)}")
    print(f"   External: Pos={len(X_ext_pos)}, Neg={0 if X_ext_neg is None else len(X_ext_neg)}")

    # Keep each model's val for later validation ROC
    X_val_list = [d["X_val"] for d in base_splits]
    y_val_list = [d["y_val"] for d in base_splits]

    return {
        "base_splits": base_splits,
        "X_test": X_test,
        "y_test": y_test,
        "X_val_list": X_val_list,
        "y_val_list": y_val_list,
        "X_ext_pos": X_ext_pos,
        "X_ext_neg": X_ext_neg,
        "pos_files": pos_files,
        "neg_files": neg_files,
        "ext_pos_files": ext_pos_files,
        "ext_neg_files": ext_neg_files
    }

def compute_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "threshold": float(threshold),
    }

    if y_true is not None:
        if len(np.unique(y_true)) >= 2:
            metrics["auc"] = float(roc_auc_score(y_true, y_proba))
        else:
            metrics["auc"] = np.nan

        metrics["accuracy"] = float(accuracy_score(y_true, y_pred)) if len(np.unique(y_true)) >= 2 else np.nan

        if np.sum(y_true == 1) > 0:
            metrics["sensitivity"] = float(recall_score(y_true, y_pred, pos_label=1))
        else:
            metrics["sensitivity"] = np.nan

        if np.sum(y_true == 0) > 0:
            metrics["specificity"] = float(recall_score(y_true, y_pred, pos_label=0))
        else:
            metrics["specificity"] = np.nan

        if np.sum(y_true == 1) > 0 and np.sum(y_pred == 1) > 0:
            metrics["f1"] = float(f1_score(y_true, y_pred))
        else:
            metrics["f1"] = np.nan

        if len(np.unique(y_true)) >= 2 and len(np.unique(y_pred)) >= 2:
            metrics["mcc"] = float(matthews_corrcoef(y_true, y_pred))
        else:
            metrics["mcc"] = np.nan
    else:
        metrics.update({
            "accuracy": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "mcc": np.nan
        })

    return metrics

def summarize_metrics(metric_list):
    df = pd.DataFrame(metric_list)
    summary = {}
    for col in df.columns:
        if col != "model":
            summary[col] = {
                "mean": float(np.nanmean(df[col])),
                "std": float(np.nanstd(df[col], ddof=1)) if len(df[col]) > 1 else 0.0
            }
    return summary

def print_summary(summary, title="Summary"):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    for col, stats in summary.items():
        print(f"{col:12s}: {stats['mean']:.4f} ± {stats['std']:.4f}")

def plot_and_save_roc(y_true, y_score, title, save_path):
    if y_true is None or len(np.unique(y_true)) < 2:
        print(f"Skipping ROC plot (insufficient label classes): {title}")
        return None

    try:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(title)
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        return roc_auc
    except Exception as e:
        print(f"ROC plot failed: {title} | error: {e}")
        plt.close()
        return None

def save_metrics_csv(save_dir, df, filename):
    df.to_csv(os.path.join(save_dir, filename), index=False)
    print(f"Saved: {filename}")

<h1>Model and optimization</h1>

In [ ]:
def create_lr_from_params(params):
    penalty = params.get("penalty", "l2")
    if penalty == "none":
        penalty = None

    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            C=params.get("C", 1.0),
            penalty=penalty,
            solver=params.get("solver", "lbfgs"),
            l1_ratio=params.get("l1_ratio", None),
            max_iter=params.get("max_iter", 1000),
            class_weight="balanced",
            random_state=RANDOM_SEED
        ))
    ])

def create_svm_from_params(params):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            C=params.get("C", 1.0),
            kernel=params.get("kernel", "rbf"),
            gamma=params.get("gamma", "scale"),
            coef0=params.get("coef0", 0.0),
            degree=params.get("degree", 3),
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            max_iter=params.get("max_iter", -1)
        ))
    ])

def create_rf_from_params(params):
    return RandomForestClassifier(
        n_estimators=params.get("n_estimators", 100),
        criterion=params.get("criterion", "gini"),
        max_depth=params.get("max_depth", None),
        min_samples_split=params.get("min_samples_split", 2),
        min_samples_leaf=params.get("min_samples_leaf", 1),
        max_features=params.get("max_features", "sqrt"),
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

def create_mlp_from_params(params):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=params.get("layer_widths", (128, 64)),
            activation=params.get("activation", "relu"),
            solver=params.get("solver", "adam"),
            alpha=params.get("alpha", 1e-4),
            learning_rate_init=params.get("learning_rate_init", 1e-3),
            max_iter=params.get("max_iter", 300),
            early_stopping=params.get("early_stopping", False),
            validation_fraction=params.get("validation_fraction", 0.0),
            n_iter_no_change=params.get("n_iter_no_change", 30),
            tol=params.get("tol", 1e-5),
            random_state=RANDOM_SEED
        ))
    ])

def create_model_from_params(model_type, params):
    if model_type == "lr":
        return create_lr_from_params(params)
    elif model_type == "svm":
        return create_svm_from_params(params)
    elif model_type == "rf":
        return create_rf_from_params(params)
    elif model_type == "mlp":
        return create_mlp_from_params(params)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
def optuna_objective_lr(X_train, y_train, trial):
    C = trial.suggest_float('C', 1e-2, 1e1, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet', 'none'])
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga'])

    if penalty == 'l1' and solver not in ['liblinear', 'saga']:
        return 0.0
    if penalty == 'elasticnet' and solver != 'saga':
        return 0.0
    if penalty == 'none' and solver not in ['lbfgs', 'newton-cg', 'sag', 'saga']:
        return 0.0

    l1_ratio = None
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.01, 0.99)

    params = {
        'C': C,
        'penalty': penalty,
        'solver': solver,
        'l1_ratio': l1_ratio,
        'max_iter': trial.suggest_int('max_iter', 500, 3000)
    }

    return crossval_auc(create_lr_from_params, params, X_train, y_train)

def optuna_objective_svm(X_train, y_train, trial):
    params = {
        'C': trial.suggest_float('C', 1e-3, 1e2, log=True),
        'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid']),
        'degree': 3,
        'gamma': 'scale',
        'coef0': 0.0
    }

    if params['kernel'] == 'poly':
        params['degree'] = trial.suggest_int('degree', 2, 5)
    if params['kernel'] in ['rbf', 'poly', 'sigmoid']:
        params['gamma'] = trial.suggest_categorical('gamma', ['scale', 'auto'])
    if params['kernel'] in ['poly', 'sigmoid']:
        params['coef0'] = trial.suggest_float('coef0', 0.0, 10.0)

    return crossval_auc(create_svm_from_params, params, X_train, y_train)

def optuna_objective_rf(X_train, y_train, trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=50),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }

    use_max_depth = trial.suggest_categorical('use_max_depth', [True, False])
    params['max_depth'] = trial.suggest_int('max_depth', 3, 20) if use_max_depth else None

    return crossval_auc(create_rf_from_params, params, X_train, y_train)

def optuna_objective_mlp(X_train, y_train, trial):
    n_layers = trial.suggest_int('n_layers', 2, 5)
    layer_widths = tuple(trial.suggest_int(f'layer_{i}_width', 50, 400) for i in range(n_layers))

    params = {
        'n_layers': n_layers,
        'layer_widths': layer_widths,
        'alpha': trial.suggest_float('alpha', 1e-6, 1e-2, log=True),
        'activation': trial.suggest_categorical('activation', ['tanh', 'relu']),
        'solver': trial.suggest_categorical('solver', ['adam', 'sgd']),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
        'max_iter': trial.suggest_int('max_iter', 200, 400),
        'early_stopping': False,
        'validation_fraction': 0.0,
        'n_iter_no_change': 30,
        'tol': 1e-5
    }

    return crossval_auc(create_mlp_from_params, params, X_train, y_train)

def crossval_auc(create_func, params, X_train, y_train):
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val_split = X_train[train_idx], X_train[val_idx]
        y_tr, y_val_split = y_train[train_idx], y_train[val_idx]

        model = create_func(params)
        try:
            model.fit(X_tr, y_tr)
            y_proba = model.predict_proba(X_val_split)[:, 1]
            auc_scores.append(roc_auc_score(y_val_split, y_proba))
        except Exception:
            return 0.0

    return float(np.mean(auc_scores))

def get_optuna_objective(model_type):
    if model_type == "lr":
        return optuna_objective_lr
    elif model_type == "svm":
        return optuna_objective_svm
    elif model_type == "rf":
        return optuna_objective_rf
    elif model_type == "mlp":
        return optuna_objective_mlp
    else:
        raise ValueError(f"Unknown model type: {model_type}")

<h1>Training functions</h1>

In [ ]:
def train_base_models(model_type, data, optuna_trials=OPTUNA_TRIALS):
    set_all_seeds(RANDOM_SEED)

    base_splits = data["base_splits"]
    X_test = data["X_test"]
    y_test = data["y_test"]

    models = []
    model_params_list = []
    studies = []
    base_model_val_metrics = []
    base_model_test_metrics = []

    objective_func = get_optuna_objective(model_type)

    for k in range(N_BASE_MODELS):
        X_train_k = base_splits[k]["X_train"]
        y_train_k = base_splits[k]["y_train"]
        X_val_k = base_splits[k]["X_val"]
        y_val_k = base_splits[k]["y_val"]

        print(f"\nTraining {model_type.upper()} base model {k+1}/{N_BASE_MODELS}")
        print(f"   Train: {int(np.sum(y_train_k==1))} Pos + {int(np.sum(y_train_k==0))} Neg  (total {len(y_train_k)})")
        print(f"   Val  : {int(np.sum(y_val_k==1))} Pos + {int(np.sum(y_val_k==0))} Neg  (total {len(y_val_k)})")

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED + k)
        )
        study.optimize(
            lambda trial: objective_func(X_train_k, y_train_k, trial),
            n_trials=optuna_trials,
            show_progress_bar=False
        )

        final_params = study.best_params.copy()
        
        if "layer_0_width" in final_params:
            n_layers = final_params.get("n_layers", 2)
            final_params["layer_widths"] = tuple(
                final_params[f"layer_{i}_width"] for i in range(n_layers)
            )
        model = create_model_from_params(model_type, final_params)
        model.fit(X_train_k, y_train_k)

        y_val_proba = model.predict_proba(X_val_k)[:, 1]
        fpr, tpr, thresholds = roc_curve(y_val_k, y_val_proba)
        youden_idx = np.argmax(tpr - fpr)
        model_threshold = float(thresholds[youden_idx])
        final_params["inference_threshold"] = model_threshold

        val_metrics = compute_metrics(y_val_k, y_val_proba, threshold=model_threshold)

        # Test: evaluate on the fixed internal test set (28+28)
        y_test_proba = model.predict_proba(X_test)[:, 1]
        test_metrics = compute_metrics(y_test, y_test_proba, threshold=model_threshold)

        models.append(model)
        model_params_list.append(final_params)
        studies.append(study)
        base_model_val_metrics.append(val_metrics)
        base_model_test_metrics.append(test_metrics)

        print(f"   Best threshold (from this model's Val): {model_threshold:.4f}")
        print(f"   Val AUC: {val_metrics['auc']:.4f} | Internal Test AUC: {test_metrics['auc']:.4f}")

    return {
        "model_type": model_type,
        "models": models,
        "model_params_list": model_params_list,
        "studies": studies,
        "base_model_val_metrics": base_model_val_metrics,
        "base_model_test_metrics": base_model_test_metrics,
        "X_test": X_test,
        "y_test": y_test,
        "X_val_list": data.get("X_val_list", None),
        "y_val_list": data.get("y_val_list", None),
    }

<h1>Evaluation functions one</h1>

In [ ]:
def evaluate_base_models(models, thresholds, X, y=None, dataset_name="dataset"):
    results = []
    proba_list = []

    print(f"\n{dataset_name} - Base model results")
    for i, model in enumerate(models):
        proba = model.predict_proba(X)[:, 1]
        proba_list.append(proba)

        metrics = compute_metrics(y, proba, threshold=thresholds[i]) if y is not None else {
            "threshold": thresholds[i],
            "accuracy": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "mcc": np.nan
        }
        metrics["model"] = f"Model_{i+1}"
        results.append(metrics)

        print(f"Model {i+1} | Thr={thresholds[i]:.4f} | "
              f"Acc={metrics['accuracy']:.4f} | Sens={metrics['sensitivity']:.4f} | "
              f"Spec={metrics['specificity']:.4f} | F1={metrics['f1']:.4f} | "
              f"AUC={metrics['auc']:.4f} | MCC={metrics['mcc']:.4f}")

    summary = summarize_metrics(results)
    print_summary(summary, title=f"{dataset_name} - 5 base models Mean ± Std")

    return pd.DataFrame(results), summary, proba_list


def evaluate_ensemble(models, thresholds, X, y=None, dataset_name="dataset"):
    proba_list = [model.predict_proba(X)[:, 1] for model in models]
    avg_proba = np.mean(proba_list, axis=0)
    avg_threshold = float(np.mean(thresholds))

    metrics = compute_metrics(y, avg_proba, threshold=avg_threshold) if y is not None else {
        "threshold": avg_threshold,
        "accuracy": np.nan,
        "sensitivity": np.nan,
        "specificity": np.nan,
        "f1": np.nan,
        "auc": np.nan,
        "mcc": np.nan
    }

    print(f"\n{dataset_name} - Probability-averaged ensemble results")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    return metrics, avg_proba, avg_threshold


def evaluate_external_dataset(result, X_ext_pos, X_ext_neg=None, dataset_name="external"):
    models = result["models"]
    thresholds = [p["inference_threshold"] for p in result["model_params_list"]]

    if X_ext_neg is not None:
        X_ext = np.concatenate([X_ext_pos, X_ext_neg], axis=0)
        y_ext = np.concatenate([np.ones(len(X_ext_pos)), np.zeros(len(X_ext_neg))])
        mode = "paired"
    else:
        X_ext = X_ext_pos
        y_ext = np.ones(len(X_ext_pos))
        mode = "pos_only"

    base_df, base_summary, proba_list = evaluate_base_models(
        models, thresholds, X_ext, y=y_ext,
        dataset_name=f"{dataset_name} ({mode})"
    )

    ensemble_metrics, avg_proba, avg_threshold = evaluate_ensemble(
        models, thresholds, X_ext, y=y_ext,
        dataset_name=f"{dataset_name} ({mode})"
    )

    return {
        "mode": mode,
        "X": X_ext,
        "y": y_ext,
        "base_df": base_df,
        "base_summary": base_summary,
        "ensemble_metrics": ensemble_metrics,
        "avg_proba": avg_proba,
        "avg_threshold": avg_threshold
    }

<h1>Save functions</h1>

In [ ]:
def save_summary_log(save_dir, config, model_params_list,
                     base_model_val_df, base_model_test_df,
                     internal_ensemble_metrics,
                     external_base_df=None, external_ensemble_metrics=None):
    log_path = os.path.join(save_dir, "summary.log")

    with open(log_path, "w", encoding="utf-8") as f:
        f.write("AFP Ensemble Model Summary Log\n")
        f.write("=" * 80 + "\n\n")

        f.write("Basic Configuration\n")
        f.write("=" * 80 + "\n")
        for k, v in config.items():
            f.write(f"{k}: {v}\n")

        f.write("\nBest Hyperparameters and Validation Thresholds per Base Model\n")
        f.write("=" * 80 + "\n")
        for i, params in enumerate(model_params_list):
            f.write(f"\n--- Model {i+1} ---\n")
            for pk, pv in params.items():
                f.write(f"{pk}: {pv}\n")

        f.write("\nValidation Set Base Model Performance\n")
        f.write("=" * 80 + "\n")
        f.write(base_model_val_df.to_string(index=False))
        f.write("\n\n")

        f.write("Internal Test Set Base Model Performance\n")
        f.write("=" * 80 + "\n")
        f.write(base_model_test_df.to_string(index=False))
        f.write("\n\n")

        f.write("Internal Test Set Probability-Averaged Ensemble Performance\n")
        f.write("=" * 80 + "\n")
        for k, v in internal_ensemble_metrics.items():
            f.write(f"{k}: {v}\n")

        if external_base_df is not None:
            f.write("\n\nExternal Test Set Base Model Performance\n")
            f.write("=" * 80 + "\n")
            f.write(external_base_df.to_string(index=False))
            f.write("\n\n")

        if external_ensemble_metrics is not None:
            f.write("External Test Set Probability-Averaged Ensemble Performance\n")
            f.write("=" * 80 + "\n")
            for k, v in external_ensemble_metrics.items():
                f.write(f"{k}: {v}\n")

    print(f"summary.log saved to: {log_path}")


def save_prediction_details(save_dir, models, model_params_list,
                            X_internal, y_internal,
                            internal_avg_proba, internal_avg_threshold,
                            X_external=None, y_external=None,
                            external_avg_proba=None, external_avg_threshold=None):
    thresholds = [params["inference_threshold"] for params in model_params_list]

    # Internal test set
    internal_df = pd.DataFrame({"y_true": y_internal.astype(int)})
    for i, model in enumerate(models):
        proba = model.predict_proba(X_internal)[:, 1]
        pred = (proba >= thresholds[i]).astype(int)
        internal_df[f"model_{i+1}_proba"] = proba
        internal_df[f"model_{i+1}_pred"] = pred
        internal_df[f"model_{i+1}_threshold"] = thresholds[i]

    internal_df["ensemble_avg_proba"] = internal_avg_proba
    internal_df["ensemble_avg_threshold"] = internal_avg_threshold
    internal_df["ensemble_avg_pred"] = (internal_avg_proba >= internal_avg_threshold).astype(int)
    internal_df.to_csv(os.path.join(save_dir, "internal_test_predictions.csv"), index=False)

    # External test set
    if X_external is not None and external_avg_proba is not None:
        external_df = pd.DataFrame({"y_true": y_external.astype(int) if y_external is not None else np.nan})
        for i, model in enumerate(models):
            proba = model.predict_proba(X_external)[:, 1]
            external_df[f"model_{i+1}_proba"] = proba
            if y_external is not None:
                external_df[f"model_{i+1}_pred"] = (proba >= thresholds[i]).astype(int)
            external_df[f"model_{i+1}_threshold"] = thresholds[i]

        external_df["ensemble_avg_proba"] = external_avg_proba
        external_df["ensemble_avg_threshold"] = external_avg_threshold
        if y_external is not None:
            external_df["ensemble_avg_pred"] = (external_avg_proba >= external_avg_threshold).astype(int)
        external_df.to_csv(os.path.join(save_dir, "external_test_predictions.csv"), index=False)

    print("prediction details CSV saved")


def save_optuna_plots(save_dir, studies):
    if not VISUALIZATION_AVAILABLE:
        print("Skipping Optuna visualization (plotly not installed)")
        return

    if studies is None or len(studies) == 0:
        print("No Study object to visualize")
        return

    optuna_plot_dir = os.path.join(save_dir, "optuna_convergence_plots")
    os.makedirs(optuna_plot_dir, exist_ok=True)

    for i, study in enumerate(studies):
        try:
            fig = vis.plot_optimization_history(study)
            fig.update_layout(title=f"Model {i+1} Optimization History (Best AUC: {study.best_value:.4f})")
            html_path = os.path.join(optuna_plot_dir, f"model_{i+1}_convergence.html")
            fig.write_html(html_path)
            print(f"Saved model {i+1} convergence plot: {html_path}")
        except Exception as e:
            print(f"Failed to save model {i+1} Optuna plot: {e}")



def save_roc_plots(save_dir, models, X_val, y_val,
                   X_internal, y_internal, internal_avg_proba, internal_ensemble_metrics,
                   X_external=None, y_external=None, external_avg_proba=None, external_ensemble_metrics=None,
                   X_val_list=None, y_val_list=None):

    roc_dir = os.path.join(save_dir, "roc_curves")
    os.makedirs(roc_dir, exist_ok=True)

    # 1) Combined AUROC plot of 5 validation base models
    try:
        plt.figure(figsize=(10, 8))
        for i, model in enumerate(models):
            if X_val_list is not None and y_val_list is not None:
                Xv, yv = X_val_list[i], y_val_list[i]
            else:
                Xv, yv = X_val, y_val

            y_score = model.predict_proba(Xv)[:, 1]
            fpr, tpr, _ = roc_curve(yv, y_score)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, label=f'Model {i+1} (AUC={roc_auc:.4f})')

        plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Validation AUROC Curves of 5 Base Models")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(os.path.join(roc_dir, "validation_5models_AUROC.png"), dpi=300, bbox_inches='tight')
        plt.close()
        print("Saved validation_5models_AUROC.png")
    except Exception as e:
        print(f"Failed to save combined validation 5-model ROC plot: {e}")
        plt.close()

    # 2) Validation ROC for each model
    for i, model in enumerate(models):
        try:
            if X_val_list is not None and y_val_list is not None:
                Xv, yv = X_val_list[i], y_val_list[i]
            else:
                Xv, yv = X_val, y_val

            y_score = model.predict_proba(Xv)[:, 1]
            plot_and_save_roc(
                yv, y_score,
                f"Base Model {i+1} - ROC (Validation)",
                os.path.join(roc_dir, f"base_model_{i+1}_val_ROC.png")
            )
        except Exception as e:
            print(f"Failed to save validation base model {i+1} ROC: {e}")

    # 3) Internal test ROC for each model
    if X_internal is not None and y_internal is not None and len(np.unique(y_internal)) >= 2:
        for i, model in enumerate(models):
            try:
                y_score = model.predict_proba(X_internal)[:, 1]
                plot_and_save_roc(
                    y_internal, y_score,
                    f"Base Model {i+1} - ROC (Internal Test)",
                    os.path.join(roc_dir, f"base_model_{i+1}_internal_test_ROC.png")
                )
            except Exception as e:
                print(f"Failed to save internal test base model {i+1} ROC: {e}")

    # 4) Internal test ensemble ROC
    try:
        plot_and_save_roc(
            y_internal, internal_avg_proba,
            f"Probability Ensemble - ROC (Internal Test)\nAUC={internal_ensemble_metrics['auc']:.4f}",
            os.path.join(roc_dir, "probability_ensemble_internal_test_ROC.png")
        )
    except Exception as e:
        print(f"Failed to save internal test ensemble ROC: {e}")

    # 5) External test ROC for each base model
    if X_external is not None and y_external is not None and len(np.unique(y_external)) >= 2:
        for i, model in enumerate(models):
            try:
                y_score = model.predict_proba(X_external)[:, 1]
                plot_and_save_roc(
                    y_external, y_score,
                    f"Base Model {i+1} - ROC (External Test)",
                    os.path.join(roc_dir, f"base_model_{i+1}_external_test_ROC.png")
                )
            except Exception as e:
                print(f"Failed to save external test base model {i+1} ROC: {e}")

    # 6) External test ensemble ROC
    if X_external is not None and y_external is not None and external_avg_proba is not None:
        try:
            plot_and_save_roc(
                y_external, external_avg_proba,
                f"Probability Ensemble - ROC (External Test)\nAUC={external_ensemble_metrics['auc']:.4f}",
                os.path.join(roc_dir, "probability_ensemble_external_test_ROC.png")
            )
        except Exception as e:
            print(f"Failed to save external test ensemble ROC: {e}")

<h1>Control functions</h1>

In [ ]:
def run_experiment(model_type, data=None, ext_pos_dir=None, ext_neg_dir=None,
                   ext_pos_expected=164, ext_neg_expected=164, save_outputs=True):

    if data is None:
        data = load_all_esm_features(
            ext_pos_dir=ext_pos_dir,
            ext_neg_dir=ext_neg_dir,
            ext_pos_expected=ext_pos_expected,
            ext_neg_expected=ext_neg_expected
        )

    result = train_base_models(model_type=model_type, data=data, optuna_trials=OPTUNA_TRIALS)
    thresholds = [p["inference_threshold"] for p in result["model_params_list"]]

    # Internal test set (fixed 28+28)
    internal_base_df, internal_base_summary, _ = evaluate_base_models(
        result["models"], thresholds, result["X_test"], y=result["y_test"], dataset_name="Internal Test"
    )
    internal_ensemble_metrics, internal_avg_proba, internal_avg_threshold = evaluate_ensemble(
        result["models"], thresholds, result["X_test"], y=result["y_test"], dataset_name="Internal Test"
    )

    result["internal_base_df"] = internal_base_df
    result["internal_base_summary"] = internal_base_summary
    result["internal_ensemble_metrics"] = internal_ensemble_metrics
    result["internal_avg_proba"] = internal_avg_proba
    result["internal_avg_threshold"] = internal_avg_threshold

    # External test set
    if data.get("X_ext_pos") is not None:
        ext_result = evaluate_external_dataset(
            result,
            X_ext_pos=data["X_ext_pos"],
            X_ext_neg=data.get("X_ext_neg", None),
            dataset_name="External Test"
        )
        result["external_result"] = ext_result

    if save_outputs:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_dir = os.path.join(MODEL_SAVE_DIR, f"AFP_Ensemble_{model_type.upper()}_ESM2_{timestamp}")
        os.makedirs(save_dir, exist_ok=True)
        result["save_dir"] = save_dir

        # 1) Save model
        model_save_path = os.path.join(save_dir, f"{model_type}_ensemble_models.pkl")
        joblib.dump({
            "model_type": model_type,
            "models": result["models"],
            "model_params_list": result["model_params_list"],
            "ensemble_avg_threshold": result["internal_avg_threshold"],
            "created_at": timestamp,
            "feature_type": f"ESM2_mean_rep_layer_{ESM_LAYER}",
            "split_logic": {
                "POS_TOTAL": POS_TOTAL,
                "POS_POOL": POS_POOL,
                "POS_TEST": POS_TEST,
                "NEG_TOTAL": NEG_TOTAL,
                "NEG_POOL": NEG_POOL,
                "NEG_TEST": NEG_TEST,
                "N_BASE_MODELS": N_BASE_MODELS,
                "N_FOLDS": N_FOLDS,
                "NEG_GROUP_SIZE": NEG_GROUP_SIZE,
                "FOLD_SIZE_POS": FOLD_SIZE_POS,
                "FOLD_SIZE_NEG": FOLD_SIZE_NEG,
            }
        }, model_save_path)
        print(f"Model saved to: {model_save_path}")

        # 2) Save metrics csv
        save_metrics_csv(save_dir, pd.DataFrame(result["base_model_val_metrics"]), "base_model_validation_metrics.csv")
        save_metrics_csv(save_dir, result["internal_base_df"], "base_model_internal_test_metrics.csv")
        pd.DataFrame([result["internal_ensemble_metrics"]]).to_csv(
            os.path.join(save_dir, "ensemble_internal_test_metrics.csv"), index=False
        )

        if "external_result" in result:
            save_metrics_csv(save_dir, result["external_result"]["base_df"], "base_model_external_metrics.csv")
            pd.DataFrame([result["external_result"]["ensemble_metrics"]]).to_csv(
                os.path.join(save_dir, "ensemble_external_metrics.csv"), index=False
            )

        # 3) Save prediction details
        save_prediction_details(
            save_dir=save_dir,
            models=result["models"],
            model_params_list=result["model_params_list"],
            X_internal=result["X_test"],
            y_internal=result["y_test"],
            internal_avg_proba=result["internal_avg_proba"],
            internal_avg_threshold=result["internal_avg_threshold"],
            X_external=result["external_result"]["X"] if "external_result" in result else None,
            y_external=result["external_result"]["y"] if "external_result" in result else None,
            external_avg_proba=result["external_result"]["avg_proba"] if "external_result" in result else None,
            external_avg_threshold=result["external_result"]["avg_threshold"] if "external_result" in result else None
        )

        # 4) Save ROC
        save_roc_plots(
            save_dir=save_dir,
            models=result["models"],
            X_val=None,
            y_val=None,
            X_val_list=result.get("X_val_list", None),
            y_val_list=result.get("y_val_list", None),
            X_internal=result["X_test"],
            y_internal=result["y_test"],
            internal_avg_proba=result["internal_avg_proba"],
            internal_ensemble_metrics=result["internal_ensemble_metrics"],
            X_external=result["external_result"]["X"] if "external_result" in result else None,
            y_external=result["external_result"]["y"] if "external_result" in result else None,
            external_avg_proba=result["external_result"]["avg_proba"] if "external_result" in result else None,
            external_ensemble_metrics=result["external_result"]["ensemble_metrics"] if "external_result" in result else None
        )

        # 5) Save Optuna plots
        save_optuna_plots(save_dir, result["studies"])

        # 6) Save summary.log
        config = {
            "RANDOM_SEED": RANDOM_SEED,
            "MODEL_TYPE": model_type.upper(),
            "FEATURE_TYPE": f"ESM2_mean_rep_layer_{ESM_LAYER}",
            "OPTUNA_TRIALS": OPTUNA_TRIALS,

            "SPLIT_LOGIC": "Pos:153 -> 125(5-fold) + 28(test); Neg:653 -> 625(5 groups*125, each 5-fold) + 28(test)",
            "N_BASE_MODELS": N_BASE_MODELS,
            "N_FOLDS": N_FOLDS,

            "POS_TOTAL": POS_TOTAL,
            "POS_POOL": POS_POOL,
            "POS_TEST": POS_TEST,
            "POS_FOLD_SIZE": FOLD_SIZE_POS,

            "NEG_TOTAL": NEG_TOTAL,
            "NEG_POOL": NEG_POOL,
            "NEG_TEST": NEG_TEST,
            "NEG_GROUP_SIZE": NEG_GROUP_SIZE,
            "NEG_FOLD_SIZE": FOLD_SIZE_NEG,

            "PER_MODEL_TRAIN": "100 Pos + 100 Neg (4 folds)",
            "PER_MODEL_VAL": "25 Pos + 25 Neg (1 fold)",
            "INTERNAL_TEST": "28 Pos + 28 Neg (fixed)"
        }

        save_summary_log(
            save_dir=save_dir,
            config=config,
            model_params_list=result["model_params_list"],
            base_model_val_df=pd.DataFrame(result["base_model_val_metrics"]),
            base_model_test_df=result["internal_base_df"],
            internal_ensemble_metrics=result["internal_ensemble_metrics"],
            external_base_df=result["external_result"]["base_df"] if "external_result" in result else None,
            external_ensemble_metrics=result["external_result"]["ensemble_metrics"] if "external_result" in result else None
        )

    return result

<h1>Evaluation functions two</h1>

In [ ]:
def load_saved_model_bundle(model_pkl_path):
    bundle = joblib.load(model_pkl_path)
    print(f"Loaded model: {model_pkl_path}")
    print(f"   model_type: {bundle.get('model_type', 'unknown')}")
    print(f"   Number of models: {len(bundle['models'])}")
    return bundle

def load_external_eval_embeddings(pos_dir, neg_dir=None, pos_expected=None, neg_expected=None, layer=ESM_LAYER):
    X_pos, pos_files = load_embeddings_from_dir(
        pos_dir, layer=layer, expected_count=pos_expected, dataset_name="Eval Positive"
    )

    X_neg, neg_files = None, None
    if neg_dir is not None:
        X_neg, neg_files = load_embeddings_from_dir(
            neg_dir, layer=layer, expected_count=neg_expected, dataset_name="Eval Negative"
        )

    return {
        "X_pos": X_pos,
        "X_neg": X_neg,
        "pos_files": pos_files,
        "neg_files": neg_files
    }


def evaluate_saved_model_bundle(model_pkl_path, pos_dir, neg_dir=None,
                                pos_expected=None, neg_expected=None,
                                dataset_name="Custom External Eval",
                                save_csv_path=None):
    bundle = load_saved_model_bundle(model_pkl_path)
    models = bundle["models"]
    params_list = bundle["model_params_list"]
    thresholds = [p["inference_threshold"] for p in params_list]
    model_type = bundle.get("model_type", "unknown")

    eval_data = load_external_eval_embeddings(
        pos_dir=pos_dir,
        neg_dir=neg_dir,
        pos_expected=pos_expected,
        neg_expected=neg_expected,
        layer=ESM_LAYER
    )

    X_pos = eval_data["X_pos"]
    X_neg = eval_data["X_neg"]

    if X_neg is not None:
        X_eval = np.concatenate([X_pos, X_neg], axis=0)
        y_eval = np.concatenate([np.ones(len(X_pos)), np.zeros(len(X_neg))])
        mode = "paired"
    else:
        X_eval = X_pos
        y_eval = np.ones(len(X_pos))
        mode = "pos_only"

    print(f"\nStarting evaluation: {dataset_name} | mode={mode}")

    base_df, base_summary, _ = evaluate_base_models(
        models=models,
        thresholds=thresholds,
        X=X_eval,
        y=y_eval,
        dataset_name=dataset_name
    )

    ensemble_metrics, avg_proba, avg_threshold = evaluate_ensemble(
        models=models,
        thresholds=thresholds,
        X=X_eval,
        y=y_eval,
        dataset_name=dataset_name
    )

    if save_csv_path is not None:
        os.makedirs(save_csv_path, exist_ok=True)

        # 1. metrics csv
        base_df.to_csv(os.path.join(save_csv_path, "reloaded_base_model_metrics.csv"), index=False)
        pd.DataFrame([ensemble_metrics]).to_csv(
            os.path.join(save_csv_path, "reloaded_ensemble_metrics.csv"), index=False
        )

        # 2. prediction csv
        pred_df = pd.DataFrame({"y_true": y_eval.astype(int)})
        for i, model in enumerate(models):
            proba = model.predict_proba(X_eval)[:, 1]
            pred = (proba >= thresholds[i]).astype(int)
            pred_df[f"model_{i+1}_proba"] = proba
            pred_df[f"model_{i+1}_pred"] = pred
            pred_df[f"model_{i+1}_threshold"] = thresholds[i]

        pred_df["ensemble_avg_proba"] = avg_proba
        pred_df["ensemble_avg_threshold"] = avg_threshold
        pred_df["ensemble_avg_pred"] = (avg_proba >= avg_threshold).astype(int)
        pred_df.to_csv(os.path.join(save_csv_path, "reloaded_predictions.csv"), index=False)

        # 3. summary.log
        reloaded_log_path = os.path.join(save_csv_path, "summary.log")
        with open(reloaded_log_path, "w", encoding="utf-8") as f:
            f.write("Reloaded Model Evaluation Summary\n")
            f.write("=" * 80 + "\n\n")
            f.write(f"model_path: {model_pkl_path}\n")
            f.write(f"model_type: {model_type}\n")
            f.write(f"dataset_name: {dataset_name}\n")
            f.write(f"mode: {mode}\n")
            f.write(f"pos_dir: {pos_dir}\n")
            f.write(f"neg_dir: {neg_dir}\n")
            f.write(f"avg_threshold: {avg_threshold}\n\n")

            f.write("Base Model Metrics\n")
            f.write("=" * 80 + "\n")
            f.write(base_df.to_string(index=False))
            f.write("\n\n")

            f.write("Ensemble Metrics\n")
            f.write("=" * 80 + "\n")
            for k, v in ensemble_metrics.items():
                f.write(f"{k}: {v}\n")

        # 4. ROC plots
        roc_dir = os.path.join(save_csv_path, "roc_curves")
        os.makedirs(roc_dir, exist_ok=True)

        if mode == "paired" and len(np.unique(y_eval)) >= 2:
            # External ROC for each base model
            for i, model in enumerate(models):
                try:
                    y_score = model.predict_proba(X_eval)[:, 1]
                    plot_and_save_roc(
                        y_eval, y_score,
                        f"Reloaded Base Model {i+1} - ROC ({dataset_name})",
                        os.path.join(roc_dir, f"reloaded_base_model_{i+1}_external_ROC.png")
                    )
                except Exception as e:
                    print(f"Failed to save Reloaded base model {i+1} ROC: {e}")

            try:
                plot_and_save_roc(
                    y_eval, avg_proba,
                    f"Reloaded Ensemble - ROC ({dataset_name})\nAUC={ensemble_metrics['auc']:.4f}",
                    os.path.join(roc_dir, "reloaded_ensemble_external_ROC.png")
                )
            except Exception as e:
                print(f"Failed to save Reloaded ensemble ROC: {e}")

        print(f"Reloaded evaluation results saved to: {save_csv_path}")

    return {
        "bundle": bundle,
        "mode": mode,
        "base_df": base_df,
        "base_summary": base_summary,
        "ensemble_metrics": ensemble_metrics,
        "avg_proba": avg_proba,
        "avg_threshold": avg_threshold
    }

<h1>Training</h1>

In [ ]:
# =========================
# Load data
# =========================
data = load_all_esm_features()

# =========================
# Train the model
# =========================
result_mlp = run_experiment("mlp", data=data, save_outputs=True)

<h1>Evaluation</h1>

In [ ]:
# ========================
# Re-evaluate saved model bundle on external datasets
# ========================
eval_result = evaluate_saved_model_bundle(
    model_pkl_path="/path/to/model_save/mlp_ensemble_models.pkl",
    pos_dir="/path/to/external_positive_esm_embedding",
    neg_dir="/path/to/external_negative_esm_embedding",
    pos_expected=164,
    neg_expected=164,
    dataset_name="External_164",
    save_csv_path="/path/to/model_save/re_eval_external_164"
)
# ========================
# Re-evaluate saved model bundle on positive-only external datasets
# ========================
eval_pos_only = evaluate_saved_model_bundle(
    model_pkl_path="/path/to/model_save/mlp_ensemble_models.pkl",
    pos_dir="/path/to/external_positive_only_esm_embedding",
    neg_dir=None,
    pos_expected=6,
    dataset_name="PositiveOnly_6",
    save_csv_path="/path/to/model_save/re_eval_positive_only_6"
)